# Superposicion MAGL humana vs. T. cruzi -- version interactiva (cealign)

Este notebook **no corre PyMOL ni recalcula ningun alineamiento**. Toda la parte
pesada (cealign, identificacion de triada, RMSD, refinamiento local) ya se hizo
en `src/case_studies/magl/superpose_magl.py` (entorno `magl_superposicion`, con
`pymol-open-source`). Esto solo **lee** los PDBs ya alineados y el resumen JSON
que ese script exporta a `paths.case_study_dir` (config/config.yaml), y arma
una visualizacion interactiva con py3Dmol -- pensado para correr en el kernel
`pocketvec_env` (Biopython + py3Dmol + ipywidgets), sin depender de pymol2.

Reemplaza a `Superposicion.ipynb`, que tenia tres problemas (ver auditoria):
1. Alineaba con Biopython (secuencia global + Kabsch), no con cealign.
2. No identificaba ni destacaba la triada catalitica.
3. Tenia paths hardcodeados de otra maquina y en realidad cargaba datos de
   GSK3-alpha, no de MAGL -- el resumen HTML decia "3PE6"/"MAGL" como texto fijo,
   sin ninguna relacion con los datos realmente cargados.

Todos los numeros y nombres del resumen final salen de `alignment_summary.json`,
nada esta tipeado a mano.


In [ ]:
import json
import os

from Bio.PDB import PDBParser, PDBIO, Select
import numpy as np
import pandas as pd
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

# Run from notebooks/: paths come from config/config.yaml (paths.case_study_dir).
import sys
sys.path.insert(0, os.path.join(os.path.abspath(".."), "src"))
from utils.config import CASE_STUDY_DIR

OUT_DIR = CASE_STUDY_DIR
SUMMARY_PATH = os.path.join(OUT_DIR, "alignment_summary.json")

with open(SUMMARY_PATH) as fh:
    summary = json.load(fh)

summary


## 1. Estructuras + triada + bolsillos (todo desde el JSON, nada hardcodeado)

`human` = fija (nunca se mueve). `cruzi` = movil, **ya transformada por cealign**
(el PDB en disco tiene las coordenadas post-alineamiento, no las originales del
modelo de AlphaFold).

In [ ]:
human_full_pdb = summary["human"]["full_pdb"]
human_pocket_pdb = summary["human"]["pocket_pdb"]
cruzi_full_pdb = summary["cruzi"]["full_pdb"]
cruzi_pocket_pdb = summary["cruzi"]["pocket_pdb"]

for label, path in [
    ("humano (fijo, completo)", human_full_pdb),
    ("humano (fijo, bolsillo)", human_pocket_pdb),
    ("cruzi (movil, completo, post-cealign)", cruzi_full_pdb),
    ("cruzi (movil, bolsillo, post-cealign)", cruzi_pocket_pdb),
]:
    print(f"{label:<42} {'OK' if os.path.exists(path) else 'FALTA'}  {path}")


In [ ]:
human_triad = summary["human"]["triad"]   # {"ser": 122, "asp": 239, "his": 269}
cruzi_triad = summary["cruzi"]["triad"]    # {"ser": 139, "asp": 256, "his": 286, "asp_resname": "ASP"}

human_triad_resnums = [human_triad["ser"], human_triad["asp"], human_triad["his"]]
cruzi_triad_resnums = [cruzi_triad["ser"], cruzi_triad["asp"], cruzi_triad["his"]]

print("Triada humana (literatura, verificada en superpose_magl.py):",
      f"Ser{human_triad['ser']} / Asp{human_triad['asp']} / His{human_triad['his']}")
print("Triada cruzi (identificada dinamicamente -- motivo GXSXG + proximidad 3D):",
      f"Ser{cruzi_triad['ser']} / {cruzi_triad.get('asp_resname','ASP')}{cruzi_triad['asp']} / His{cruzi_triad['his']}")


## 2. Residuos de cada bolsillo (adaptado de `Superposicion.ipynb`)

Misma logica que el notebook viejo (`get_pocket_residues_from_file`): un
residuo representativo por atomo CA (o el primer atomo si no tiene CA),
para poder despues buscar coincidencias por distancia.

In [ ]:
_parser = PDBParser(QUIET=True)


def get_pocket_residues_from_file(pdb_path):
    """Adaptado de Superposicion.ipynb (get_pocket_residues_from_structure /
    get_pocket_residues_from_file) -- misma logica, aplicada aca a los PDBs
    ya exportados por superpose_magl.py."""
    structure = _parser.get_structure("pocket", pdb_path)
    residues = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.id[0] != " ":
                    continue
                atom = residue["CA"] if residue.has_id("CA") else next(residue.get_atoms())
                residues.append({
                    "chain": chain.id,
                    "res_id": residue.id[1],
                    "resname": residue.get_resname(),
                    "coord": atom.get_coord(),
                })
    return residues


residues_human_pocket = get_pocket_residues_from_file(human_pocket_pdb)
residues_cruzi_pocket = get_pocket_residues_from_file(cruzi_pocket_pdb)

print(f"Bolsillo humano ({summary['human']['centroid_source']}) -- {len(residues_human_pocket)} residuos:")
for r in residues_human_pocket:
    print(f"  {r['chain']} {r['res_id']:>4} {r['resname']}")

print(f"\nBolsillo cruzi ({summary['cruzi']['centroid_source']}) -- {len(residues_cruzi_pocket)} residuos:")
for r in residues_cruzi_pocket:
    print(f"  {r['chain']} {r['res_id']:>4} {r['resname']}")


## 3. Coincidencias residuo-a-residuo entre ambos bolsillos (distancia CA < 4 A)

Misma funcion `find_aligned_residues` del notebook viejo, sin cambios en la
logica -- lo unico que cambia es que ahora corre sobre coordenadas alineadas
con cealign en vez de Kabsch. Esto es un chequeo grueso (1 atomo CA por
residuo); la seccion siguiente hace la clasificacion fina atomo-por-atomo.

In [ ]:
def find_aligned_residues(residues_a, residues_b, cutoff=4.0):
    pairs = []
    for res_a in residues_a:
        for res_b in residues_b:
            dist = np.linalg.norm(res_a["coord"] - res_b["coord"])
            if dist < cutoff:
                pairs.append((res_a, res_b, dist))
    pairs.sort(key=lambda x: x[2])
    return pairs


aligned_pairs = find_aligned_residues(residues_human_pocket, residues_cruzi_pocket, cutoff=4.0)

print(f"Coincidencias estructurales por CA (distancia < 4 A): {len(aligned_pairs)}\n")
for res_a, res_b, dist in aligned_pairs:
    print(f"humano {res_a['chain']}:{res_a['res_id']} {res_a['resname']}  <->  "
          f"cruzi {res_b['chain']}:{res_b['res_id']} {res_b['resname']}  dist={dist:.2f} A")

if not aligned_pairs:
    print("Ninguna -- ver la clasificacion atomo-por-atomo de la seccion siguiente para el detalle fino.")


## 4. Clasificacion de solapamiento real entre bolsillos (atomo por atomo)

La comparacion por CA de arriba es gruesa (1 punto por residuo). Aca se
clasifica CADA atomo de cada bolsillo (mismos PDBs `*_pocket*.pdb`, con todos
los atomos de los residuos de bolsillo, no solo CA) segun si tiene algun atomo
del OTRO bolsillo a menos de `OVERLAP_THRESHOLD_A` (parametro configurable,
default 2.5 A -- una distancia tipica de contacto de van der Waals entre
atomos pesados). Esto da 3 grupos: **solo humano**, **solo cruzi**,
**compartido/solapado**.

In [ ]:
OVERLAP_THRESHOLD_A = 2.5  # configurable: distancia para considerar que dos atomos "se tocan"


def classify_pocket_overlap(pdb_a, pdb_b, threshold=OVERLAP_THRESHOLD_A):
    """Devuelve, para cada lado, la ESTRUCTURA parseada (no solo la lista de
    atomos) junto con la mascara booleana de solapamiento. Se devuelve la
    estructura para poder despues filtrar/guardar subconjuntos con
    PDBIO+Select sobre el MISMO objeto -- atom.get_full_id() incluye el id
    interno de la Structure, asi que comparar full_ids entre dos parses
    distintos del mismo archivo (dos llamadas a get_structure) nunca
    coincide, aunque el contenido sea identico."""
    structure_a = _parser.get_structure("a", pdb_a)
    structure_b = _parser.get_structure("b", pdb_b)
    atoms_a = list(structure_a.get_atoms())
    atoms_b = list(structure_b.get_atoms())
    coords_a = np.array([a.get_coord() for a in atoms_a])
    coords_b = np.array([a.get_coord() for a in atoms_b])
    dmat = np.linalg.norm(coords_a[:, None, :] - coords_b[None, :, :], axis=-1)
    a_shared = dmat.min(axis=1) < threshold
    b_shared = dmat.min(axis=0) < threshold
    return structure_a, structure_b, atoms_a, atoms_b, a_shared, b_shared


structure_h, structure_c, atoms_h, atoms_c, h_shared_mask, c_shared_mask = classify_pocket_overlap(
    human_pocket_pdb, cruzi_pocket_pdb
)

n_h, n_h_shared = len(atoms_h), int(h_shared_mask.sum())
n_c, n_c_shared = len(atoms_c), int(c_shared_mask.sum())

print(f"Umbral de solapamiento: {OVERLAP_THRESHOLD_A} A\n")
print(f"Bolsillo humano: {n_h} atomos totales")
print(f"  - compartido/solapado (< {OVERLAP_THRESHOLD_A} A de un atomo de cruzi): "
      f"{n_h_shared} ({100*n_h_shared/n_h:.0f}%)")
print(f"  - solo humano (sin atomo de cruzi cerca): {n_h - n_h_shared} ({100*(n_h - n_h_shared)/n_h:.0f}%)")
print()
print(f"Bolsillo cruzi: {n_c} atomos totales")
print(f"  - compartido/solapado (< {OVERLAP_THRESHOLD_A} A de un atomo humano): "
      f"{n_c_shared} ({100*n_c_shared/n_c:.0f}%)")
print(f"  - solo cruzi (sin atomo humano cerca): {n_c - n_c_shared} ({100*(n_c - n_c_shared)/n_c:.0f}%)")


In [ ]:
class AtomFullIdSelect(Select):
    """Filtra atomos por su full_id (identidad estable DENTRO de la misma
    Structure ya parseada) -- se usa para separar cada bolsillo en sus
    subconjuntos solo/compartido sin tener que reconstruir residuos a mano.
    Importante: hay que pasarle el MISMO objeto Structure del que salieron
    los atomos de `atoms`/`mask` (ver classify_pocket_overlap) -- si se
    vuelve a parsear el archivo de cero, los full_id ya no coinciden."""

    def __init__(self, keep_ids):
        self.keep_ids = keep_ids

    def accept_atom(self, atom):
        return atom.get_full_id() in self.keep_ids


def save_atom_subset(structure, atoms, mask, keep_shared, out_path):
    keep_ids = {a.get_full_id() for a, m in zip(atoms, mask) if bool(m) == keep_shared}
    if not keep_ids:
        return None
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_path, AtomFullIdSelect(keep_ids))
    return out_path


OVERLAP_DIR = os.path.join(OUT_DIR, "overlap")
os.makedirs(OVERLAP_DIR, exist_ok=True)

human_only_pdb = save_atom_subset(structure_h, atoms_h, h_shared_mask, False,
                                   os.path.join(OVERLAP_DIR, "human_pocket_only.pdb"))
human_shared_pdb = save_atom_subset(structure_h, atoms_h, h_shared_mask, True,
                                     os.path.join(OVERLAP_DIR, "human_pocket_shared.pdb"))
cruzi_only_pdb = save_atom_subset(structure_c, atoms_c, c_shared_mask, False,
                                   os.path.join(OVERLAP_DIR, "cruzi_pocket_only.pdb"))
cruzi_shared_pdb = save_atom_subset(structure_c, atoms_c, c_shared_mask, True,
                                     os.path.join(OVERLAP_DIR, "cruzi_pocket_shared.pdb"))

for label, path in [("humano - solo", human_only_pdb), ("humano - compartido", human_shared_pdb),
                     ("cruzi - solo", cruzi_only_pdb), ("cruzi - compartido", cruzi_shared_pdb)]:
    print(f"{label:<24} -> {path if path else '(vacio, no se genero PDB)'}")


## 5. Visualizacion py3Dmol superpuesta (3 colores de bolsillo + triada + toggles)

Cambios respecto a la version anterior de este notebook:
- Los bolsillos ya NO son 2 superficies (magenta/cian) que se ven "adyacentes
  pero no queda claro donde solapan" -- ahora son **3** superficies: magenta
  = solo humano, cian = solo cruzi, **verde = compartido/solapado** (seccion 4).
- Las 6 etiquetas de la triada (3 humanas + 3 cruzi) usan `screenOffset` de
  3Dmol.js -- un offset en PIXELES de pantalla, no en coordenadas 3D, asi que
  quedan separadas sin importar el angulo de camara (el offset en 3D que
  usabamos antes solo separaba bien desde ciertos angulos). Se agrega ademas
  una tabla HTML con el detalle debajo del visor, como respaldo.
- Checkboxes (ipywidgets) para togglear cada capa por separado: proteina
  humana, proteina cruzi, bolsillo humano, bolsillo cruzi, bolsillo
  compartido, triada humana, triada cruzi. Funcionan con un kernel vivo
  (JupyterLab/VS Code) -- si se abre como HTML estatico exportado, los
  checkboxes se ven pero no responden (no hay kernel corriendo del otro lado).

In [ ]:
viewer = py3Dmol.view(width=1000, height=700)
viewer.setBackgroundColor("white")

MODEL = {}  # nombre de capa -> indice de modelo en el viewer

def _add(name, pdb_path):
    if pdb_path is None:
        MODEL[name] = None
        return
    viewer.addModel(open(pdb_path).read(), "pdb")
    MODEL[name] = len(MODEL_ORDER)
    MODEL_ORDER.append(name)

MODEL_ORDER = []
_add("protein_human", human_full_pdb)
_add("protein_cruzi", cruzi_full_pdb)
_add("pocket_human_only", human_only_pdb)
_add("pocket_cruzi_only", cruzi_only_pdb)
_add("pocket_human_shared", human_shared_pdb)
_add("pocket_cruzi_shared", cruzi_shared_pdb)

CARTOON_COLOR = {"protein_human": "#b0b0b0", "protein_cruzi": "#555555"}
SURFACE_COLOR = {
    "pocket_human_only": "magenta", "pocket_cruzi_only": "cyan",
    "pocket_human_shared": "lime", "pocket_cruzi_shared": "lime",
}

state = {
    "protein_human": True, "protein_cruzi": True,
    "pocket_human_only": True, "pocket_cruzi_only": True, "pocket_shared": True,
    "triad_human": True, "triad_cruzi": True,
}


def refresh(_=None):
    # --- cartoons (base) ---
    for name in ("protein_human", "protein_cruzi"):
        idx = MODEL[name]
        viewer.setStyle({"model": idx}, {"cartoon": {"color": CARTOON_COLOR[name]}} if state[name] else {})

    # --- triada: sticks encima del cartoon (solo si la proteina de ese lado esta prendida) ---
    if MODEL["protein_human"] is not None:
        on = state["protein_human"] and state["triad_human"]
        sel = {"model": MODEL["protein_human"], "resi": human_triad_resnums}
        viewer.setStyle(sel, {"stick": {"colorscheme": "yellowCarbon", "radius": 0.28}} if on
                         else ({"cartoon": {"color": CARTOON_COLOR["protein_human"]}} if state["protein_human"] else {}))
    if MODEL["protein_cruzi"] is not None:
        on = state["protein_cruzi"] and state["triad_cruzi"]
        sel = {"model": MODEL["protein_cruzi"], "resi": cruzi_triad_resnums}
        viewer.setStyle(sel, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.28}} if on
                         else ({"cartoon": {"color": CARTOON_COLOR["protein_cruzi"]}} if state["protein_cruzi"] else {}))

    # --- superficies de bolsillo: rebuild completo (mas simple/robusto que remove/add incremental) ---
    viewer.removeAllSurfaces()
    if state["pocket_human_only"] and MODEL["pocket_human_only"] is not None:
        viewer.addSurface(py3Dmol.VDW, {"opacity": 0.85, "color": SURFACE_COLOR["pocket_human_only"]},
                           {"model": MODEL["pocket_human_only"]})
    if state["pocket_cruzi_only"] and MODEL["pocket_cruzi_only"] is not None:
        viewer.addSurface(py3Dmol.VDW, {"opacity": 0.85, "color": SURFACE_COLOR["pocket_cruzi_only"]},
                           {"model": MODEL["pocket_cruzi_only"]})
    if state["pocket_shared"]:
        for name in ("pocket_human_shared", "pocket_cruzi_shared"):
            if MODEL[name] is not None:
                viewer.addSurface(py3Dmol.VDW, {"opacity": 0.9, "color": SURFACE_COLOR[name]}, {"model": MODEL[name]})

    # --- labels de triada: screenOffset (separacion en pixeles, no en 3D) ---
    viewer.removeAllLabels()
    label_offsets = {"ser": (-60, -55), "asp": (-60, 0), "his": (-60, 55)}
    if state["protein_human"] and state["triad_human"]:
        for key, resnum in [("ser", human_triad["ser"]), ("asp", human_triad["asp"]), ("his", human_triad["his"])]:
            dx, dy = label_offsets[key]
            viewer.addLabel(f"{key.upper()}{resnum} (human)", {
                "screenOffset": {"x": -dx, "y": dy},
                "backgroundColor": "black", "fontColor": "yellow", "fontSize": 12,
            }, {"model": MODEL["protein_human"], "resi": resnum, "atom": "CA"})
    if state["protein_cruzi"] and state["triad_cruzi"]:
        for key, resnum in [("ser", cruzi_triad["ser"]), ("asp", cruzi_triad["asp"]), ("his", cruzi_triad["his"])]:
            dx, dy = label_offsets[key]
            viewer.addLabel(f"{key.upper()}{resnum} (cruzi)", {
                "screenOffset": {"x": dx, "y": dy},
                "backgroundColor": "black", "fontColor": "orange", "fontSize": 12,
            }, {"model": MODEL["protein_cruzi"], "resi": resnum, "atom": "CA"})

    viewer.render()


refresh()
viewer.zoomTo({"model": [MODEL[n] for n in ("pocket_human_only", "pocket_cruzi_only",
                                             "pocket_human_shared", "pocket_cruzi_shared") if MODEL[n] is not None]})
viewer.show()


In [ ]:
# Tabla de respaldo (no-flotante) con el detalle de la triada -- por si el
# visor 3D queda muy chico o rotado para leer los labels.
triad_rows = [
    ("Ser", human_triad["ser"], cruzi_triad["ser"]),
    (cruzi_triad.get("asp_resname", "Asp").capitalize(), human_triad["asp"], cruzi_triad["asp"]),
    ("His", human_triad["his"], cruzi_triad["his"]),
]
triad_df = pd.DataFrame(triad_rows, columns=["Posicion catalitica", "Humano (3PE6)", "Cruzi (AF-Q4CSM0-F1)"])
display(triad_df)


In [ ]:
# Checkboxes de toggle por capa. Requieren un kernel vivo para responder
# (en un HTML exportado estatico se ven pero no hacen nada).
checkbox_specs = [
    ("protein_human", "Proteina humana (cartoon)"),
    ("protein_cruzi", "Proteina cruzi (cartoon)"),
    ("pocket_human_only", "Bolsillo humano -- solo humano (magenta)"),
    ("pocket_cruzi_only", "Bolsillo cruzi -- solo cruzi (cian)"),
    ("pocket_shared", "Bolsillo -- compartido/solapado (verde)"),
    ("triad_human", "Triada humana (sticks amarillos)"),
    ("triad_cruzi", "Triada cruzi (sticks naranjas)"),
]

checkboxes = []
for key, label in checkbox_specs:
    cb = widgets.Checkbox(value=state[key], description=label, indent=False,
                           layout=widgets.Layout(width="380px"))

    def _on_change(change, key=key):
        state[key] = change["new"]
        refresh()

    cb.observe(_on_change, names="value")
    checkboxes.append(cb)

display(widgets.VBox(checkboxes))


## 6. Vista lado a lado (sin superponer, misma orientacion de camara)

Dos visores py3Dmol independientes, uno por proteina+bolsillo. Ninguno de los
dos llama a `.rotate()`, y como `cruzi` ya esta en el mismo sistema de
coordenadas que `human` (por el cealign previo), `zoomTo()` sobre la region de
bolsillo+triada de cada lado produce el MISMO angulo de camara en ambos
visores -- lo unico que puede diferir es el centrado/zoom exacto segun la
extension propia de cada bolsillo.

In [ ]:
def build_single_viewer(full_pdb, only_pdb, shared_pdb, triad_resnums, cartoon_color,
                         only_color, triad_colorscheme, chain_id, triad_dict, width=480, height=550):
    v = py3Dmol.view(width=width, height=height)
    v.setBackgroundColor("white")

    v.addModel(open(full_pdb).read(), "pdb")
    v.setStyle({"model": 0}, {"cartoon": {"color": cartoon_color}})
    v.setStyle({"model": 0, "resi": triad_resnums}, {"stick": {"colorscheme": triad_colorscheme, "radius": 0.28}})

    zoom_models = [0]
    if only_pdb is not None:
        v.addModel(open(only_pdb).read(), "pdb")
        v.addSurface(py3Dmol.VDW, {"opacity": 0.85, "color": only_color}, {"model": 1})
        zoom_models.append(1)
    if shared_pdb is not None:
        v.addModel(open(shared_pdb).read(), "pdb")
        v.addSurface(py3Dmol.VDW, {"opacity": 0.9, "color": "lime"}, {"model": len(zoom_models)})
        zoom_models.append(len(zoom_models))

    v.zoomTo({"model": zoom_models})
    return v


viewer_human = build_single_viewer(
    human_full_pdb, human_only_pdb, human_shared_pdb, human_triad_resnums,
    cartoon_color="#b0b0b0", only_color="magenta", triad_colorscheme="yellowCarbon",
    chain_id=summary["human"]["chain"], triad_dict=human_triad,
)
viewer_cruzi = build_single_viewer(
    cruzi_full_pdb, cruzi_only_pdb, cruzi_shared_pdb, cruzi_triad_resnums,
    cartoon_color="#555555", only_color="cyan", triad_colorscheme="orangeCarbon",
    chain_id=summary["cruzi"]["chain"], triad_dict=cruzi_triad,
)

print(f"Izquierda: humano ({summary['human']['code']})  |  Derecha: cruzi ({summary['cruzi']['code']})")

# Se arma un unico output HTML con flexbox (en vez de widgets.Output/HBox) para
# que ambos visores salgan como texto/html plano, mas simple y robusto para
# exportar/inspeccionar que envolverlos en widgets de ipywidgets.
combined_html = f"""
<div style="display:flex; gap:12px; align-items:flex-start;">
  <div style="flex:1;">
    <div style="text-align:center; font-weight:bold; margin-bottom:4px;">Humano (fijo)</div>
    {viewer_human._make_html()}
  </div>
  <div style="flex:1;">
    <div style="text-align:center; font-weight:bold; margin-bottom:4px;">Cruzi (movil, post-cealign)</div>
    {viewer_cruzi._make_html()}
  </div>
</div>
"""
display(HTML(combined_html))


## 7. Resumen final (generado 100% desde `alignment_summary.json`)

A diferencia del notebook viejo (que mostraba "3PE6"/"MAGL" como texto fijo sin
relacion con los datos cargados), todo lo que sigue sale de `summary` (mas los
porcentajes de solapamiento calculados en la seccion 4) -- si se corre este
notebook para otro par humano-cruzi, el texto cambia solo.

In [ ]:
html = f"""
<div style="font-family:Arial; color:white; background:#111; padding:14px; border-radius:8px; line-height:1.5;">
  <b>Comparacion estructural (cealign, con triada catalitica anclada)</b><br>
  <b>Proteina fija:</b> {summary['human']['code']} (cadena {summary['human']['chain']})<br>
  <b>Proteina movil:</b> {summary['cruzi']['code']} (cadena {summary['cruzi']['chain']})<br><br>

  <b>Bolsillo fijo (humano):</b> centroide de {summary['human']['centroid_source']}<br>
  <b>Druggability Score (fpocket) fijo:</b> {summary['human']['fpocket_druggability_score']:.3f}<br>
  <b>Docking suitability (0-1) fijo:</b> {summary['human']['docking_suitability_score_0_1']:.2f}<br><br>

  <b>Bolsillo movil (cruzi):</b> {summary['cruzi']['centroid_source']}<br>
  <b>Druggability Score (fpocket) movil:</b> {summary['cruzi']['fpocket_druggability_score']:.3f}<br><br>

  <b>Triada humana:</b> Ser{summary['human']['triad']['ser']} /
    Asp{summary['human']['triad']['asp']} / His{summary['human']['triad']['his']}<br>
  <b>Triada cruzi:</b> Ser{summary['cruzi']['triad']['ser']} /
    {summary['cruzi']['triad'].get('asp_resname','Asp')}{summary['cruzi']['triad']['asp']} /
    His{summary['cruzi']['triad']['his']}<br><br>

  <b>RMSD global (cealign, backbone completo):</b> {summary['rmsd']['new_method']['global_A']:.3f} A
    ({summary['rmsd']['new_method']['n_atoms']} posiciones)<br>
  <b>RMSD triada catalitica (post-cealign):</b> {summary['rmsd']['new_method']['triad_A']:.3f} A<br><br>

  <b>Solapamiento atomo-a-atomo de bolsillo (umbral {OVERLAP_THRESHOLD_A} A):</b><br>
  &nbsp;&nbsp;humano: {n_h_shared}/{n_h} atomos compartidos ({100*n_h_shared/n_h:.0f}%)<br>
  &nbsp;&nbsp;cruzi:  {n_c_shared}/{n_c} atomos compartidos ({100*n_c_shared/n_c:.0f}%)<br>
</div>
"""

display(HTML(html))


## 8. Comparacion viejo (Kabsch) vs. nuevo (cealign) y conclusion

In [ ]:
old = summary["rmsd"]["old_method"]
new = summary["rmsd"]["new_method"]
refined = summary["rmsd"]["new_method_refined"]

rows = [
    (old["name"], old["global_A"], old["triad_A"], old["n_atoms"]),
    (new["name"], new["global_A"], new["triad_A"], new["n_atoms"]),
]
if refined is not None:
    rows.append((refined["name"], refined["global_A"], refined["triad_A"], refined["n_atoms"]))

table_html = "<table style='border-collapse:collapse;'>"
table_html += ("<tr><th style='text-align:left;padding:4px 10px;'>Metodo</th>"
               "<th style='padding:4px 10px;'>RMSD global (A)</th>"
               "<th style='padding:4px 10px;'>RMSD triada (A)</th>"
               "<th style='padding:4px 10px;'>N atomos</th></tr>")
for name, g, t, n in rows:
    table_html += (f"<tr><td style='padding:4px 10px;'>{name}</td>"
                    f"<td style='padding:4px 10px;text-align:right;'>{g:.3f}</td>"
                    f"<td style='padding:4px 10px;text-align:right;'>{t:.3f}</td>"
                    f"<td style='padding:4px 10px;text-align:right;'>{n}</td></tr>")
table_html += "</table>"
display(HTML(table_html))

delta_global = old["global_A"] - new["global_A"]
delta_triad = old["triad_A"] - new["triad_A"]
triad_vs_global_new = "mejor" if new["triad_A"] <= new["global_A"] else "peor"
overlap_avg_pct = 100 * (n_h_shared / n_h + n_c_shared / n_c) / 2
shift_conclusion = (
    f"el bolsillo humano YA NO se ve claramente desplazado respecto al de cruzi en la triada "
    f"(ancla mas confiable, RMSD {new['triad_A']:.2f} A), pero solo un {overlap_avg_pct:.0f}% de los "
    f"atomos de bolsillo en promedio quedan a <{OVERLAP_THRESHOLD_A} A de un atomo del otro lado -- "
    f"la mayoria del contorno del bolsillo sigue sin solapar, compatible con una diferencia real de "
    f"forma entre especies, no con un artefacto de alineamiento"
    if new["triad_A"] < 1.5 else
    "el desplazamiento del bolsillo humano sigue siendo visible incluso con cealign -- no era "
    "solo un artefacto del metodo viejo"
)

display(Markdown(f"""
**cealign vs. Biopython+Kabsch, para {summary['human']['code']} (humano) vs. {summary['cruzi']['code']} (cruzi):**

- RMSD global: {old['global_A']:.3f} A (viejo) &rarr; {new['global_A']:.3f} A (nuevo) &mdash;
  {'mejora' if delta_global > 0 else 'empeora'} en {abs(delta_global):.3f} A.
- RMSD de triada catalitica: {old['triad_A']:.3f} A (viejo) &rarr; {new['triad_A']:.3f} A (nuevo) &mdash;
  {'mejora' if delta_triad > 0 else 'empeora'} en {abs(delta_triad):.3f} A, y queda **{triad_vs_global_new}**
  que el promedio del resto de la proteina.
- Solapamiento atomo-a-atomo del bolsillo completo (umbral {OVERLAP_THRESHOLD_A} A): {overlap_avg_pct:.0f}% en promedio.

**Conclusion:** {shift_conclusion}.
"""))
